In [1]:
# Imports for Cross-Encoder models

import os, gc, time, copy, wandb, random, logging, warnings

from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import polars as pl

from kaggle_secrets import UserSecretsClient

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold, GroupKFold

from umap import UMAP
from umap.utils import disconnected_vertices

from hdbscan import HDBSCAN

import transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

E0000 00:00:1780595807.723001      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780595807.832523      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780595808.751713      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780595808.751761      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780595808.751764      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780595808.751767      58 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
# Load datasets and encode label as `label_id`

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

OPTION_COLS = ["A", "B", "C", "D", "E"]

LABEL_COL = "answer"
LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_data = pl.read_csv(TRAIN_PATH).with_row_index("_idx")
test_data = pl.read_csv(TEST_PATH).with_row_index("_idx")
train_data = train_data.with_columns(
    pl.col(LABEL_COL).replace(LABEL2ID).cast(pl.Int8).alias("label_id")
)

ids = train_data["id"].to_list()
prompts = train_data["prompt"].to_list()

In [ ]:
# Configure data, models, device, and seeds for reproducibility

EPOCHS = 7
N_SPLITS = 5

ID_COL = "id"
QUESTION_COL = "prompt"
OPTION_COLS  = ["A", "B", "C", "D", "E"]

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

RANK_MODEL = "cross-encoder/ms-marco-MiniLM-L12-v2"
# RANK_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
# RANK_MODEL = "cross-encoder/ms-marco-MiniLM-L2-v2"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

def set_seed(seed: int):
    
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# Create directory to store saved models

BEST_DIR = "/kaggle/working/best-cv-models"
os.makedirs(BEST_DIR, exist_ok=True)

# Configure logging levels to hide model-loading report

hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

# Configure WandB info

ist_now = datetime.now(ZoneInfo("Asia/Kolkata"))
WANDB_RUN_NAME = f"{RANK_MODEL.split('/')[-1]}_{ist_now:%Y-%m-%d_%H-%M-%S}_IST"

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

In [ ]:
# Define DataLoader seeding function, and Generator for reproducibility

def seed_worker(worker_id):
    
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)
val_generator = torch.Generator()
val_generator.manual_seed(SEED)

test_generator = torch.Generator()
test_generator.manual_seed(SEED)

In [ ]:
# # Procompute context for whole train and test data

# def add_context(df):
#     return df.with_columns(
#         pl.struct([QUESTION_COL] + OPTION_COLS).map_elements(
#             lambda row: "Question: " + str(row[QUESTION_COL]) + " Options: " + " | ".join(
#                 f"{c}) {str(row[c])}" for c in OPTION_COLS
#             ),
#             return_dtype=pl.Utf8
#         ).alias("context")
#     )

# train_data = add_context(train_data)
# test_data = add_context(test_data)

In [ ]:
# # Define Dataset class

# class PairMCQDDataset(Dataset):
    
#     def __init__(self, df, tokenizer, max_length=384):
#         texts1 = []
#         texts2 = []
#         labels = []

#         for row in df.iter_rows(named=True):
#             context = row["context"]
            
#             for c in OPTION_COLS:
#                 texts1.append(context)
#                 texts2.append(f"{c}) {str(row[c])}")
#                 labels.append(1.0 if c == row[LABEL_COL] else 0.0)

#         enc = tokenizer(
#             texts1,
#             texts2,
#             truncation=True,
#             padding=True,
#             max_length=max_length,
#             return_tensors="pt"
#         )

#         self.encodings = enc
#         self.labels = torch.tensor(labels, dtype=torch.float32)

#     def __len__(self):
#         return len(self.labels)

#     def __getitem__(self, idx):
#         item = {k: v[idx] for k, v in self.encodings.items()}
#         item["labels"] = self.labels[idx]
#         return item

In [ ]:
# Define Dataset class

class BatchMCQDataset(Dataset):
    
    def __init__(self, df, tokenizer, max_length=384, has_labels=True):
        has_token_type_ids = "token_type_ids" in tokenizer.model_input_names

        input_ids = []
        attention_masks = []
        token_type_ids = [] if has_token_type_ids else None
        labels = []

        for row in df.iter_rows(named=True):
            prompt = str(row["prompt"])
            choices = [str(row[c]) for c in OPTION_COLS]

            enc = tokenizer(
                [prompt] * len(OPTION_COLS),
                choices,
                truncation=True,
                padding="max_length",
                max_length=max_length,
                return_tensors="pt",
            )

            input_ids.append(enc["input_ids"])
            attention_masks.append(enc["attention_mask"])

            if has_token_type_ids:
                token_type_ids.append(enc["token_type_ids"])

            if has_labels:
                labels.append(LABEL2ID[row["answer"]])

        self.input_ids = torch.stack(input_ids)
        self.attention_mask = torch.stack(attention_masks)
        self.token_type_ids = torch.stack(token_type_ids) if has_token_type_ids else None
        self.labels = torch.tensor(labels, dtype=torch.long) if has_labels else None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }

        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[idx]

        if self.labels is not None:
            item["labels"] = self.labels[idx]

        return item

In [ ]:
# Define function to compute Mean Average Precision@3 (MAP@3)

def compute_map3(scores, df):
    scores = np.asarray(scores).reshape(len(df), 5)
    true = df["label_id"].to_numpy()

    top3 = np.argsort(-scores, axis=1)[:, :3]
    hit = top3 == true[:, None]
    ranks = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, 0)

    out = np.zeros_like(ranks, dtype=float)
    np.divide(1.0, ranks, out=out, where=ranks > 0)
    
    return float(out.mean())

In [ ]:
# # Define function to return test prediction scores given a cross-encoder model

# def get_test_scores(model):
#     tmp = test_data.with_columns(pl.lit("A").alias("answer"))
    
#     test_loader = DataLoader(
#         PairMCQDDataset(tmp, tokenizer),
#         batch_size=32,
#         shuffle=False,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=seed_worker,
#         generator=test_generator
#     )
    
#     model.eval()
#     scores_all = []

#     with torch.no_grad():
#         for batch in test_loader:
#             labels = batch["labels"].to(DEVICE, non_blocking=True)
#             inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}

#             logits = model(**inputs).logits.squeeze(-1)
#             scores_all.extend(logits.detach().cpu().numpy().tolist())

#     return np.array(scores_all).reshape(len(test_data), 5)

In [ ]:
# Define function to return test prediction scores given a cross-encoder model

def get_batch_test_scores(model):
    tmp = test_data.clone()
    
    test_loader = DataLoader(
        BatchMCQDataset(
            tmp,
            tokenizer=tokenizer,
            max_length=384,
            has_labels=False,
        ),
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=test_generator,
    )

    model.eval()
    scores_all = []

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

            B, C, L = inputs["input_ids"].shape

            flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}
            logits = model(**flat_inputs).logits.squeeze(-1).view(B, C)

            scores_all.append(logits.detach().cpu().numpy())

    return np.concatenate(scores_all, axis=0)

In [ ]:
# # Define function to compute out-of-fold scores

# def compute_oof_scores(model, loader, DEVICE):
#     model.eval()
#     scores = []

#     with torch.no_grad():
#         for batch in loader:
#             inputs = {
#                 k: v.to(DEVICE, non_blocking=True)
#                 for k, v in batch.items()
#                 if k != "labels"
#             }
#             logits = model(**inputs).logits.squeeze(-1)
#             scores.extend(logits.detach().cpu().numpy().tolist())

#     return np.array(scores)

In [ ]:
# Define function to compute out-of-fold scores

def compute_batch_oof_scores(model, loader, DEVICE):
    model.eval()
    scores = []

    with torch.no_grad():
        for batch in loader:
            inputs = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }

            B, C, L = inputs["input_ids"].shape
            flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}
            logits = model(**flat_inputs).logits.squeeze(-1).view(B, C)

            scores.append(logits.detach().cpu().numpy())

    return np.concatenate(scores, axis=0)

In [ ]:
# # Configure CV constructor and Tokenizer

# kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# tokenizer = AutoTokenizer.from_pretrained(RANK_MODEL)

# oof_scores = np.zeros((len(train_data), 5))
# fold_best_paths = []

# # Configure WandB logging

# run = wandb.init(
#     entity="24f2005537-dl-genai-project",
#     project="dl-genai-project",
#     config={
#         "base_model": RANK_MODEL,
#         "architecture": "BERT",
#         "total_parameters": "~33.4M",
#         "transformer_layers": 12,
#         "hidden_size": 384,
#         "max_sequence_length": "512_tokens",
#         "total_splits": N_SPLITS,
#         "epochs_p_split": EPOCHS,
#         "optimizer": "AdamW",
#         "lr": 2e-5,
#         "loss": "CrossEntropyLoss",
#         "scheduler": None
#     },
# )

# # Start Training Loop

# for fold, (train_idx, val_idx) in enumerate(kf.split(train_data)):
#     fold_start = time.perf_counter()
#     print(f"\nFOLD {fold+1}/{N_SPLITS}...")

#     train_fold = train_data[train_idx]
#     val_fold = train_data[val_idx]

#     # Configure DataLoaders for current fold
    
#     train_loader = DataLoader(
#         PairMCQDDataset(train_fold, tokenizer),
#         batch_size=32,
#         shuffle=True,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=seed_worker,
#         generator=train_generator
#     )

#     val_loader = DataLoader(
#         PairMCQDDataset(val_fold, tokenizer),
#         batch_size=32,
#         shuffle=False,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=seed_worker,
#         generator=val_generator
#     )

#     ce = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)
    
#     if torch.cuda.device_count() > 1:
#         ce = nn.DataParallel(ce, device_ids=[0, 1])
#     ce = ce.to(DEVICE)

#     optimizer = torch.optim.AdamW(
#         ce.module.parameters() if isinstance(ce, nn.DataParallel) else ce.parameters(),
#         lr=2e-5,
#         weight_decay=0.025
#     )
#     loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

#     best_val_map3 = -1.0
#     best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
#     fold_best_paths.append(best_path)

#     # Srart training for {EPOCHS} in current fold
    
#     for epoch in range(EPOCHS):
#         epoch_start = time.perf_counter()

#         ce.train()
#         train_loss_sum = 0.0
#         train_logits_all = []
#         train_labels_all = []

#         for batch in train_loader:
#             labels = batch["labels"].to(DEVICE, non_blocking=True)
#             inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}

#             optimizer.zero_grad()
#             logits = ce(**inputs).logits.squeeze(-1)
#             loss = loss_fn(logits, labels)

#             loss.backward()
#             optimizer.step()

#             train_loss_sum += loss.item() * labels.size(0)
#             train_logits_all.extend(logits.detach().cpu().numpy().tolist())
#             train_labels_all.extend(labels.detach().cpu().numpy().tolist())

#         train_loss = train_loss_sum / len(train_fold)
#         train_map3 = compute_map3(train_logits_all, train_fold)

#         ce.eval()
#         val_loss_sum = 0.0
#         val_logits_all = []
#         val_labels_all = []

#         with torch.no_grad():
#             for batch in val_loader:
#                 labels = batch["labels"].to(DEVICE, non_blocking=True)
#                 inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}

#                 logits = ce(**inputs).logits.squeeze(-1)
#                 loss = loss_fn(logits, labels)

#                 val_loss_sum += loss.item() * labels.size(0)
#                 val_logits_all.extend(logits.detach().cpu().numpy().tolist())
#                 val_labels_all.extend(labels.detach().cpu().numpy().tolist())

#         val_loss = val_loss_sum / len(val_fold)
#         val_map3 = compute_map3(val_logits_all, val_fold)

#         epoch_time = time.perf_counter() - epoch_start

#         run.log({
#             "epoch_secs": round(epoch_time, 4),
#             "train_loss": train_loss,
#             "train_map3": train_map3,
#             "val_loss": val_loss,
#             "val_map3": val_map3
#         })
#         print(
#             f"Fold: {fold+1}/{N_SPLITS} "
#             f"Epoch: {epoch+1}/{EPOCHS} "
#             f"Time to complete: {epoch_time:.2f}s\n"
#             f"\tTrain Loss: {train_loss:.6f} Train MAP@3: {train_map3:.6f}\n"
#             f"\tVal Loss: {val_loss:.6f} Val MAP@3: {val_map3:.6f}"
#         )

#         if val_map3 > best_val_map3:
#             best_val_map3 = val_map3
#             state = ce.module.state_dict() if isinstance(ce, nn.DataParallel) else ce.state_dict()
#             torch.save(state, best_path)

#     best_ce = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)
#     state = torch.load(best_path, map_location="cpu")
#     best_ce.load_state_dict(state)
    
#     if torch.cuda.device_count() > 1:
#         best_ce = nn.DataParallel(best_ce, device_ids=[0, 1])
    
#     best_ce = best_ce.to(DEVICE)

#     best_val_logits = compute_oof_scores(best_ce, val_loader, DEVICE)
#     oof_scores[val_idx] = best_val_logits.reshape(len(val_fold), 5)
    
#     fold_time = time.perf_counter() - fold_start

#     run.log({
#         "fold_secs": round(fold_time, 4),
#         "best_val_map3": best_val_map3
#     })
#     print(f"\nFold {fold+1} completed in {int(fold_time//60)} min {fold_time%60:.2f}s"
#           f" | Best Val MAP@3: {best_val_map3:.6f}")

#     del ce, best_ce, optimizer
#     torch.cuda.empty_cache()
#     gc.collect()

# oof_map3 = compute_map3(oof_scores.flatten(), train_data)
# print(f"\nOut-of-fold MAP@3 score: {oof_map3:.8f}")

In [ ]:
# Get embeddings for dimensionality reduction and clustering

embed_model = SentenceTransformer(EMBED_MODEL).to(DEVICE)
embeddings = embed_model.encode(prompts, batch_size=32)

del embed_model
gc.collect()

In [ ]:
# Reduce high-dimensional embeddings to 10 dimensions for clustering

umap_model = UMAP(
    n_components=10,
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    n_jobs=1,
    random_state=SEED
)

umap_data = umap_model.fit_transform(embeddings)

disconnected = disconnected_vertices(umap_model)
valid_indices = np.where(~disconnected)[0]

umap_data = umap_data[valid_indices]

valid_ids = np.array(ids)[valid_indices]
valid_prompts = np.array(prompts)[valid_indices]

In [ ]:
# Find clusters from 10 dimensional embeddings for better cross-validation

clusterer_umap = HDBSCAN(
    min_cluster_size=7,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels_umap = clusterer_umap.fit_predict(umap_data)

valid_clusters = np.array(cluster_labels_umap)

In [ ]:
# Add cluster info to train datafrane

cluster_data = pl.DataFrame({
    "id": valid_ids,
    "prompt": valid_prompts,
    "cluster": valid_clusters,
})

train_data = train_data.join(
    cluster_data.select("id", "cluster"),
    on="id",
    how="left"
)

In [ ]:
# Configure CV constructor, dummy features, and Tokenizer

# kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

X = np.zeros(len(train_data))
y = np.zeros(len(train_data))
groups = train_data["cluster"].fill_null(-1).to_numpy()

tokenizer = AutoTokenizer.from_pretrained(RANK_MODEL)

oof_scores = np.zeros((len(train_data), 5), dtype=np.float32)
fold_best_paths = []

# Configure WandB logging

run = wandb.init(
    entity="24f2005537-dl-genai-project",
    project="dl-genai-project",
    name=WANDB_RUN_NAME,
    config={
        "model": {
            "name": RANK_MODEL,
            "architecture": "BERT",
            "total_parameters": "~33.4M",
            "transformer_layers": 12,
            "hidden_size": 384,
            "max_sequence_length": 512,
        },
        "training": {
            "splits": N_SPLITS,
            "epochs": EPOCHS,
            "optimizer": "AdamW",
            "lr": 2e-5,
            "loss": "CrossEntropyLoss",
            "scheduler": None
        },
        "data": {
            "batch_size": 32,
            "max_length": 384
        }
    },
)

# Start Training Loop

# for fold, (train_idx, val_idx) in enumerate(kf.split(train_data)):
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    fold_start = time.perf_counter()
    print(f"FOLD {fold+1}/{N_SPLITS} | "
          f"Train Set Size: {len(train_idx)} | Val Set Size: {len(val_idx)}\n")

    train_fold = train_data[train_idx]
    val_fold = train_data[val_idx]

    # Configure DataLoaders for current fold
    
    train_loader = DataLoader(
        BatchMCQDataset(
            train_fold,
            tokenizer=tokenizer,
            max_length=384,
            has_labels=True
        ),
        batch_size=32,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=train_generator,
    )

    val_loader = DataLoader(
        BatchMCQDataset(
            val_fold,
            tokenizer=tokenizer,
            max_length=384,
            has_labels=True
        ),
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=val_generator,
    )

    ce = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)
    
    if torch.cuda.device_count() > 1:
        ce = nn.DataParallel(ce, device_ids=[0, 1])
    ce = ce.to(DEVICE)

    optimizer = torch.optim.AdamW(
        ce.module.parameters() if isinstance(ce, nn.DataParallel) else ce.parameters(),
        lr=2e-5,
        weight_decay=0.01,
    )

    loss_fn = nn.CrossEntropyLoss()

    best_val_map3 = -1.0
    best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
    fold_best_paths.append(best_path)

    for epoch in range(EPOCHS):
        epoch_start = time.perf_counter()

        ce.train()
        train_loss_sum = 0.0

        for batch in train_loader:
            labels = batch["labels"].to(DEVICE, non_blocking=True)
            inputs = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }

            B, C, L = inputs["input_ids"].shape

            flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}

            optimizer.zero_grad()

            logits = ce(**flat_inputs).logits.squeeze(-1).view(B, C)
            loss = loss_fn(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * B

        train_loss = train_loss_sum / len(train_fold)

        ce.eval()
        val_loss_sum = 0.0
        val_logits_all = []

        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].to(DEVICE, non_blocking=True)
                inputs = {
                    k: v.to(DEVICE, non_blocking=True)
                    for k, v in batch.items()
                    if k != "labels"
                }

                B, C, L = inputs["input_ids"].shape
                flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}

                logits = ce(**flat_inputs).logits.squeeze(-1).view(B, C)
                loss = loss_fn(logits, labels)

                val_loss_sum += loss.item() * B
                val_logits_all.append(logits.detach().cpu().numpy())

        val_logits_all = np.concatenate(val_logits_all, axis=0)
        val_loss = val_loss_sum / len(val_fold)
        val_map3 = compute_map3(val_logits_all, val_fold)

        epoch_time = time.perf_counter() - epoch_start

        run.log({
            "epoch_secs": round(epoch_time, 4),
            "train_loss": train_loss,
            "val_loss": val_loss, "val_map3": val_map3
        })
        print(
            f"Fold: {fold+1}/{N_SPLITS} "
            f"Epoch: {epoch+1}/{EPOCHS} "
            f"Time: {epoch_time:.2f}s\n"
            f"\tTrain Loss: {train_loss:.6f}\n"
            f"\tVal Loss: {val_loss:.6f} | Val MAP@3: {val_map3:.6f}"
        )

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            state = ce.module.state_dict() if isinstance(ce, nn.DataParallel) else ce.state_dict()
            torch.save(state, best_path)

        del ce
        gc.collect()

    best_ce = AutoModelForSequenceClassification.from_pretrained(
        RANK_MODEL,
        num_labels=1
    )
    state = torch.load(best_path, map_location="cpu")
    best_ce.load_state_dict(state)

    if torch.cuda.device_count() > 1:
        best_ce = nn.DataParallel(best_ce, device_ids=[0, 1])

    best_ce = best_ce.to(DEVICE)

    best_val_logits = compute_batch_oof_scores(best_ce, val_loader, DEVICE)
    oof_scores[val_idx] = best_val_logits

    fold_time = time.perf_counter() - fold_start

    run.log({
        "fold_secs": round(fold_time, 4),
        "best_val_map3": best_val_map3
    })
    print(
        f"\nFold {fold+1} completed in {int(fold_time // 60)} min {fold_time % 60:.2f}s"
        f" | Best Val MAP@3: {best_val_map3:.6f}\n"
    )

    del best_ce, optimizer
    torch.cuda.empty_cache()
    gc.collect()

run.finish()
oof_map3 = compute_map3(oof_scores, train_data)
print(f"\nOut-of-fold MAP@3 score: {oof_map3:.8f}")

In [ ]:
test_scores = np.zeros((len(test_data), 5))

for fold, best_path in enumerate(fold_best_paths):
    ce = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)
    state = torch.load(best_path, map_location="cpu")
    ce.load_state_dict(state)

    if torch.cuda.device_count() > 1:
        ce = nn.DataParallel(ce, device_ids=[0, 1])
    ce = ce.to(DEVICE)

    test_scores += get_batch_test_scores(ce) / len(fold_best_paths)

    del ce
    torch.cuda.empty_cache()
    gc.collect()

top3_idx = np.argsort(-test_scores, axis=1)[:, :3]
pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

submission = pl.DataFrame({"ID": test_data[ID_COL], "Prediction": pred_strings})
submission.write_csv("submission.csv")
print(submission.sample(5))